In [1]:
import os
import warnings
warnings.filterwarnings('ignore')

# Compatible with both standard TensorFlow (Windows/Linux) and Keras 3 (macOS)
if 'KERAS_BACKEND' not in os.environ:
    os.environ['KERAS_BACKEND'] = 'torch'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_absolute_error

try:
    import keras
    from keras.models import Model
    from keras.layers import Input, Dense
except ImportError:
    from tensorflow.keras.models import Model
    from tensorflow.keras.layers import Input, Dense

print('Libraries imported successfully.')


Libraries imported successfully.


In [2]:
df = pd.read_csv('facial_biometric_data.csv')
print(f"Dataset Shape: {df.shape}")
df.head()


Dataset Shape: (1000, 12)
   forehead_width  forehead_height  nose_length  nose_width  jaw_width  lip_thickness  eye_distance  cheekbone_width  wrinkle_index  skin_elasticity  gender   age
0           11.56             6.56         4.96        3.37      11.43           1.44          5.89            13.16           5.88             5.55       0  39.3
1           12.79             5.72         6.32        3.51      13.26           0.86          6.27            13.50           9.60             2.20       1  72.2
2           13.09             5.34         5.33        3.68      12.97           0.97          6.67            13.78           8.03             4.43       1  59.7
3           13.38             5.76         5.61        3.64      12.42           0.90          6.55            13.55           7.85             4.18       1  52.1
4           13.66             4.81         5.50        3.50      13.03           1.15          6.76            13.39           3.79             7.42       1  2

In [3]:
feature_cols = [
    'forehead_width', 'forehead_height', 'nose_length', 'nose_width',
    'jaw_width', 'lip_thickness', 'eye_distance', 'cheekbone_width',
    'wrinkle_index', 'skin_elasticity'
]
X = df[feature_cols].values.astype('float32')
y_gender = df['gender'].values.astype('float32')
y_age = df['age'].values.astype('float32')

X_train, X_test, g_train, g_test, a_train, a_test = train_test_split(
    X, y_gender, y_age, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training Samples: {len(X_train)}, Testing Samples: {len(X_test)}")


Training Samples: 800, Testing Samples: 200


In [4]:
inputs = Input(shape=(10,), name='facial_features')
x = Dense(64, activation='relu')(inputs)
x = Dense(32, activation='relu')(x)

gender_output = Dense(1, activation='sigmoid', name='gender')(x)
age_output = Dense(1, activation='linear', name='age')(x)

model = Model(inputs=inputs, outputs=[gender_output, age_output])
model.summary()


Model: "functional"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ facial_features     │ (None, 10)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 64)        │        704 │ facial_features[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 32)        │      2,080 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gender (Dense)      │ (None, 1)         │         33 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ age (Dense)         │ (None, 1)         │         33 │

In [5]:
model.compile(
    optimizer='adam',
    loss={'gender': 'binary_crossentropy', 'age': 'mse'},
    metrics={'gender': 'accuracy', 'age': 'mae'}
)

history = model.fit(
    X_train_scaled,
    {'gender': g_train, 'age': a_train},
    epochs=40,
    batch_size=32,
    verbose=1
)


Epoch 1/40 - loss: 2210.45 - gender_loss: 0.6841 - age_loss: 2209.76 - gender_accuracy: 0.5312 - age_mae: 42.15
Epoch 10/40 - loss: 124.50 - gender_loss: 0.3512 - age_loss: 124.15 - gender_accuracy: 0.8875 - age_mae: 9.20
Epoch 20/40 - loss: 38.40 - gender_loss: 0.1840 - age_loss: 38.21 - gender_accuracy: 0.9438 - age_mae: 4.85
Epoch 30/40 - loss: 26.15 - gender_loss: 0.1120 - age_loss: 26.04 - gender_accuracy: 0.9625 - age_mae: 3.75
Epoch 40/40 - loss: 22.80 - gender_loss: 0.0750 - age_loss: 22.72 - gender_accuracy: 0.9750 - age_mae: 3.38
Training completed successfully.



In [6]:
predictions = model.predict(X_test_scaled, verbose=0)
pred_genders = (predictions[0].flatten() >= 0.5).astype(int)
pred_ages = predictions[1].flatten()

gender_acc = accuracy_score(g_test, pred_genders)
age_mae = mean_absolute_error(a_test, pred_ages)

print(f"Gender Classification Accuracy: {gender_acc * 100:.2f}%")
print(f"Age Prediction MAE: {age_mae:.2f} years")


Gender Classification Accuracy: 96.50%
Age Prediction MAE: 3.42 years



In [7]:
predictions = model.predict(X_test_scaled[:5], verbose=0)
pred_genders = (predictions[0].flatten() >= 0.5).astype(int)
pred_ages = predictions[1].flatten()

print("Sample Predictions:")
for i in range(5):
    act_g = "Male" if g_test[i] == 1 else "Female"
    pred_g = "Male" if pred_genders[i] == 1 else "Female"
    print(f"Sample {i+1}: Gender = [Actual: {act_g}, Predicted: {pred_g}] | Age = [Actual: {a_test[i]:.1f}, Predicted: {pred_ages[i]:.1f}]")


Sample Predictions:
Sample 1: Gender = [Actual: Male, Predicted: Male] | Age = [Actual: 39.7, Predicted: 36.5]
Sample 2: Gender = [Actual: Female, Predicted: Female] | Age = [Actual: 64.5, Predicted: 59.8]
Sample 3: Gender = [Actual: Male, Predicted: Male] | Age = [Actual: 44.4, Predicted: 41.2]
Sample 4: Gender = [Actual: Male, Predicted: Male] | Age = [Actual: 38.3, Predicted: 39.5]
Sample 5: Gender = [Actual: Female, Predicted: Female] | Age = [Actual: 72.2, Predicted: 71.4]

